# Сбор обучающего датасета (training dataset) для прогноза нагрузки (workload)

- target (target) для обучения (training): `workload_fact` из `data/processed/workload_from_history.csv`
- target (target) для оценки (evaluation): `workload_manual_pct` из `data/processed/excel_svod_workload.csv`
- features (features): whitelist (whitelist) из `calculate_parameter` + лаги (lags) + rolling stats (rolling stats) + сезонность (seasonality)

Результаты:
- `data/processed/workload_features.csv`
- `data/processed/workload_training_dataset.csv`



In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

WORKLOAD_DB_PATH = PROCESSED_DIR / "workload_from_history.csv"
EXCEL_SVOD_PATH = PROCESSED_DIR / "excel_svod_workload.csv"

RAW_FILES = [
    RAW_DIR / "data-2022.csv",
    RAW_DIR / "data-2023.csv",
    RAW_DIR / "data-2024.csv",
    RAW_DIR / "data-2025.csv",
]

# whitelist (whitelist) признаков (features) из calculate_parameter
WHITELIST_KEYS = [
    "workDayMonth_2",
    "hourInWeek_12",
    "effective_13",
    "absenteeism_14",
    "employFact_7",
    "cross_5",
    "overtime_4",
    "sla_15",
]

WORKLOAD_DB_PATH, EXCEL_SVOD_PATH, RAW_FILES


In [ ]:
# Загрузка DB (database) и Excel (Excel) датасетов

df_db = pd.read_csv(WORKLOAD_DB_PATH, parse_dates=["period"])
df_excel = pd.read_csv(EXCEL_SVOD_PATH, parse_dates=["month"])

df_db.head(), df_excel.head()


In [ ]:
# Парсинг whitelist (whitelist) признаков (features) из calculate_parameter (calculate_parameter)

import re


def to_bool(series: pd.Series) -> pd.Series:
    return series.astype(str).str.lower().isin(["true", "t", "1"])


def parse_division_title(div_json: str):
    if pd.isna(div_json):
        return None
    try:
        obj = json.loads(div_json)
        return obj.get("name")
    except Exception:
        return None


def normalize_title(name: str):
    if pd.isna(name):
        return None
    s = str(name).strip().lower()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[\"'“”«»]", "", s)
    return s


def extract_whitelist_features(calc_json: str, keys):
    if pd.isna(calc_json):
        return {}
    try:
        obj = json.loads(calc_json)
    except Exception:
        return {}

    out = {}
    for k in keys:
        v = obj.get(k)
        if isinstance(v, dict):
            out[k] = v.get("value")
        else:
            out[k] = None
    return out


rows = []
for p in RAW_FILES:
    part = pd.read_csv(p, dtype=str)
    part["period"] = pd.to_datetime(part["period"])

    # фильтр "согласовано" (approved) как в check_status_validate (check_status_validate)
    part["send"] = to_bool(part["send"])
    part["cancel_approve"] = to_bool(part["cancel_approve"])
    part["validate_one"] = to_bool(part["validate_one"])
    part["validate_two"] = to_bool(part["validate_two"])
    part["forecast"] = to_bool(part["forecast"])
    part["validate_count"] = part["validate_count"].astype(float).fillna(0).astype(int)

    mask_wait_second = (part["validate_count"] == 2) & part["validate_one"] & (~part["validate_two"])
    mask_not_approved = (~part["send"]) | part["cancel_approve"] | mask_wait_second

    part_ok = part[(~mask_not_approved) & (part["forecast"])].copy()

    part_ok["division_title"] = part_ok["division_name"].apply(parse_division_title)
    part_ok["division_title_norm"] = part_ok["division_title"].apply(normalize_title)

    for _, r in part_ok.iterrows():
        feats = extract_whitelist_features(r.get("calculate_parameter"), WHITELIST_KEYS)
        rows.append({
            "resist_id": r.get("resist_id"),
            "division_title": r.get("division_title"),
            "division_title_norm": r.get("division_title_norm"),
            "period": r.get("period"),
            **feats,
        })


df_feat = pd.DataFrame(rows)

# типы (types)
df_feat["resist_id"] = pd.to_numeric(df_feat["resist_id"], errors="coerce").astype("Int64")
for k in WHITELIST_KEYS:
    df_feat[k] = pd.to_numeric(df_feat[k], errors="coerce")

df_feat = df_feat.dropna(subset=["resist_id", "period"])

df_feat.head(), df_feat.shape


In [ ]:
# Merge (merge) target (target) + features (features) + Excel manual (manual)

# DB target (target)
df_db["resist_id"] = pd.to_numeric(df_db["resist_id"], errors="coerce").astype("Int64")
df_db = df_db.dropna(subset=["resist_id", "period"])

# Соединяем по resist_id + period
# (division_title берём из features, т.к. df_db может быть на уровне resist_id)
df = df_db.merge(df_feat, on=["resist_id", "period"], how="left", suffixes=("", "_feat"))

# Гарантируем наличие division_title / division_title_norm
if "division_title" not in df.columns:
    if "division_title_feat" in df.columns:
        df["division_title"] = df["division_title_feat"]
    else:
        df["division_title"] = None

if "division_title_norm" not in df.columns or df["division_title_norm"].isna().all():
    df["division_title_norm"] = df["division_title"].apply(normalize_title)

# Нормализуем period до месяца

df["period"] = df["period"].dt.to_period("M").dt.to_timestamp()

# Подтягиваем Excel manual (manual) с приоритетом по resist_id из префикса

df_excel_small = df_excel.rename(columns={"month": "period"})[
    ["division_title", "period", "workload_ml_pct", "workload_manual_pct"]
].copy()

# нормализация period до месяца

df_excel_small["period"] = df_excel_small["period"].dt.to_period("M").dt.to_timestamp()

# нормализация названия

df_excel_small["division_title_norm"] = df_excel_small["division_title"].apply(normalize_title)

# извлекаем resist_id из префикса "число: ..."

df_excel_small["excel_resist_id"] = (
    df_excel_small["division_title"]
    .astype(str)
    .str.extract(r"^\s*(\d+)\s*:", expand=False)
)

df_excel_small["excel_resist_id"] = pd.to_numeric(df_excel_small["excel_resist_id"], errors="coerce").astype("Int64")

# 1) merge по resist_id + period там, где excel_resist_id найден

df_excel_with_id = df_excel_small.dropna(subset=["excel_resist_id"]).copy()
df = df.merge(
    df_excel_with_id.rename(columns={"excel_resist_id": "resist_id"}),
    on=["resist_id", "period"],
    how="left",
    suffixes=("", "_excel"),
)

# 2) fallback merge по division_title_norm там, где Excel не подцепился по id

mask_no_excel = df["workload_manual_pct"].isna()
if mask_no_excel.any():
    df_excel_no_id = df_excel_small[df_excel_small["excel_resist_id"].isna()].copy()
    df = df.merge(
        df_excel_no_id[["division_title_norm", "period", "workload_ml_pct", "workload_manual_pct"]],
        on=["division_title_norm", "period"],
        how="left",
        suffixes=("", "_excel_fallback"),
    )

    # если после fallback появились значения — подставляем
    df.loc[mask_no_excel, "workload_ml_pct"] = df.loc[mask_no_excel, "workload_ml_pct_excel_fallback"]
    df.loc[mask_no_excel, "workload_manual_pct"] = df.loc[mask_no_excel, "workload_manual_pct_excel_fallback"]

    # чистим временные колонки
    df = df.drop(columns=[c for c in df.columns if c.endswith("_excel_fallback")])

# Time features (time features)
df["month_num"] = df["period"].dt.month
df["year_num"] = df["period"].dt.year
df["quarter_num"] = df["period"].dt.quarter

# Lags (lags) по resist_id
df = df.sort_values(["resist_id", "period"])
for lag in [1, 2, 3, 12]:
    df[f"lag_{lag}"] = df.groupby("resist_id")["workload_fact"].shift(lag)

# Rolling stats (rolling stats) по resist_id (используем только прошлое через shift(1))
df["roll_med_6"] = df.groupby("resist_id")["workload_fact"].transform(
    lambda s: s.shift(1).rolling(6, min_periods=3).median()
)
df["roll_mean_6"] = df.groupby("resist_id")["workload_fact"].transform(
    lambda s: s.shift(1).rolling(6, min_periods=3).mean()
)

# Для удобства
feature_cols = WHITELIST_KEYS + [
    "month_num", "year_num", "quarter_num",
    "lag_1", "lag_2", "lag_3", "lag_12",
    "roll_med_6", "roll_mean_6",
]

(df.head(), df.shape, df[feature_cols].isna().mean().sort_values(ascending=False).head(10))

# Оценка совпадений (match rate) Excel manual

total_rows = len(df)
manual_non_null = df["workload_manual_pct"].notna().sum()
match_rate = manual_non_null / total_rows if total_rows else np.nan

print(f"Excel manual coverage (coverage): {match_rate:.2%} ({manual_non_null}/{total_rows})")

# Отчёт по несовпавшим названиям (unmatched divisions)

# гарантируем колонки перед отчётом
if "division_title" not in df.columns:
    if "division_title_feat" in df.columns:
        df["division_title"] = df["division_title_feat"]
    else:
        df["division_title"] = None

if "division_title_norm" not in df.columns or df["division_title_norm"].isna().all():
    df["division_title_norm"] = df["division_title"].apply(normalize_title)

db_titles = (
    df.reindex(columns=["division_title", "division_title_norm"])
    .dropna()
    .drop_duplicates()
)

excel_titles = (
    df_excel_small.reindex(columns=["division_title", "division_title_norm"])
    .dropna()
    .drop_duplicates()
)

# Что есть в Excel, но не найдено в DB
excel_only = excel_titles[~excel_titles["division_title_norm"].isin(db_titles["division_title_norm"])]

# Что есть в DB, но не найдено в Excel
db_only = db_titles[~db_titles["division_title_norm"].isin(excel_titles["division_title_norm"]) ]

print("Excel-only (not matched in DB):", len(excel_only))
print("DB-only (not matched in Excel):", len(db_only))

display(excel_only.sort_values("division_title").head(50))
display(db_only.sort_values("division_title").head(50))


In [ ]:
# Сохранение (export) датасетов (datasets) в CSV (CSV)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

feat_out = PROCESSED_DIR / "workload_features.csv"
train_out = PROCESSED_DIR / "workload_training_dataset.csv"

df_feat.to_csv(feat_out, index=False)
df.to_csv(train_out, index=False)

feat_out, train_out
